# VGG16 Training on CIFAR-10 - Hybrid FP8 Training with torchao

Notebook ini melatih VGG16 pada CIFAR-10 menggunakan strategi presisi hibrida teroptimasi untuk GPU modern (NVIDIA Blackwell sm_120 / Ada Lovelace sm_89):
1. **BF16 Conv2d (Native)**: Layer konvolusi berjalan di bawah `torch.autocast(dtype=torch.bfloat16)` untuk menghilangkan overhead memori dan waktu dari spatial unfolding (im2col).
2. **FP8 Linear (torchao)**: Layer Fully Connected classifier head (`fc1` & `fc2`) dikonversi secara otomatis ke `Float8Linear` menggunakan library **`torchao`** (`torchao.float8`).
3. **Filter Dimensi**: Konversi FP8 dibatasi hanya untuk layer linear yang dimensinya (in_features & out_features) habis dibagi 16 (persyaratan hardware `torch._scaled_mm`). Layer output classifier (10 kelas) tetap berjalan di BF16/FP32.
4. **Memory Alignment Padding**: Output akhir classifier dipad ke 16 kelas (agar Tensor Core FP8 menyala) lalu dipotong kembali ke 10 kelas sebelum masuk loss computation.

In [ ]:
import torch
import time
import os

# Menonaktifkan TF32 agar eksperimen presisi terkontrol
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Menggunakan device: {device}")

In [ ]:
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# Standarisasi Transform Data CIFAR-10
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=0, pin_memory=True)

In [ ]:
import torch.nn as nn

# Arsitektur VGG16 Standar disesuaikan untuk CIFAR-10 (10 Kelas) dengan output classifier padded ke 16
class VGG16_CIFAR10_Padded(nn.Module):
    def __init__(self, num_classes=10):
        super(VGG16_CIFAR10_Padded, self).__init__()
        self.features = self._make_layers()
        self.classifier = nn.Sequential(
            nn.Linear(512, 4096),
            nn.ReLU(True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(True),
            nn.Dropout(),
            # Output dipad ke 16 (kelipatan 16) agar Tensor Core FP8 menyala
            nn.Linear(4096, 16)
        )
        self.num_classes = num_classes

    def forward(self, x):
        out = self.features(x)
        out = out.view(out.size(0), -1)
        logits = self.classifier(out)
        
        # Slicing metadata-only secara efisien ke 10 kelas asli
        return logits[:, :self.num_classes]

    def _make_layers(self):
        cfg = [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 'M', 512, 512, 512, 'M', 512, 512, 512, 'M']
        layers = []
        in_channels = 3
        for x in cfg:
            if x == 'M':
                layers += [nn.MaxPool2d(kernel_size=2, stride=2)]
            else:
                layers += [
                    nn.Conv2d(in_channels, x, kernel_size=3, padding=1),
                    nn.BatchNorm2d(x),
                    nn.ReLU(True)
                ]
                in_channels = x
        return nn.Sequential(*layers)

In [ ]:
def evaluate(model, data_loader, criterion, device):
    model.eval()
    correct = 0
    total = 0
    running_loss = 0.0
    with torch.no_grad():
        for inputs, targets in data_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
            
    accuracy = 100.0 * correct / total
    avg_loss = running_loss / total
    return accuracy, avg_loss

In [ ]:
from torchao.float8 import convert_to_float8_training

# 1. Inisialisasi model standar
model = VGG16_CIFAR10_Padded().to(device)

# 2. Filter untuk mengonversi layer Linear dengan dimensi kelipatan 16
def module_filter_fn(module, fqn):
    if isinstance(module, nn.Linear):
        if module.in_features % 16 == 0 and module.out_features % 16 == 0:
            return True
    return False

# 3. Konversi layer Linear terpilih ke FP8 via torchao
print("Mengonversi classifier ke Float8Linear...")
convert_to_float8_training(model, module_filter_fn=module_filter_fn)

# Tampilkan status layer untuk memastikan konversi berhasil
from torchao.float8.float8_linear import Float8Linear
fp8_layers = [name for name, m in model.named_modules() if isinstance(m, Float8Linear)]
print(f"Layer FP8 yang aktif: {fp8_layers}")

# 4. Inisialisasi Criterion dan Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
epochs = 5

In [ ]:
print("Memulai pelatihan dengan Hybrid FP8 (Conv BF16 + FC FP8)...")
for epoch in range(1, epochs + 1):
    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats(device)
        
    start_time = time.time()
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass dibungkus bfloat16 autocast (conv berjalan di BF16, FC berjalan di FP8)
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
        
    epoch_time = time.time() - start_time
    train_acc = 100.0 * correct / total
    train_loss = running_loss / total
    
    # Evaluasi test accuracy
    test_acc, test_loss = evaluate(model, test_loader, criterion, device)
    
    # Profiling GPU VRAM
    if device.type == 'cuda':
        peak_memory = torch.cuda.max_memory_allocated(device) / (1024 * 1024)
        print(f"Epoch [{epoch}/{epochs}] - Time: {epoch_time:.2f}s - "
              f"Train Loss: {train_loss:.4f} - Train Acc: {train_acc:.2f}% - "
              f"Test Loss: {test_loss:.4f} - Test Acc: {test_acc:.2f}% - "
              f"Peak VRAM: {peak_memory:.2f} MB")
    else:
        print(f"Epoch [{epoch}/{epochs}] - Time: {epoch_time:.2f}s - "
              f"Train Loss: {train_loss:.4f} - Train Acc: {train_acc:.2f}% - "
              f"Test Loss: {test_loss:.4f} - Test Acc: {test_acc:.2f}% - "
              f"Peak VRAM: N/A (CPU)")